# Session 11 - Fine-tuning Encoder-Decoders & Decoders, and How Models Decode

Last session we built sequence models **from scratch** and then met the Hugging Face `Trainer`.
Today we climb one more rung: we **fine-tune pretrained Transformers** for two different jobs, and
then open up the single most practical black box in text generation - **decoding**.

By the end you will be able to:

1. Fine-tune an **encoder-decoder** model (**T5**) on a real *summarization* task.
2. Fine-tune a **decoder-only** model (**distilgpt2**) as a *causal language model*.
3. Explain and control **greedy / beam / sampling / temperature / top-k / top-p** decoding - and implement them by hand.
4. Do zero-shot and few-shot **prompting**, and see why an *instruction-tuned* model behaves so differently.

> ### 📌 Previously, in Session 10 (*Classification, Tagging, Fine-tuning & LoRA*)
>
> A quick recap so this session stands on the same ground:
>
> - **From-scratch sequence models.** We built a **BiLSTM classifier** (sentiment on SST-2) and a **BiLSTM tagger**
>   (named-entity recognition on CoNLL) by hand: our own vocab, `torch.Dataset`, padding + the `-100` ignore-index,
>   and a **manual training loop** (`forward → loss → backward → step`).
> - **Tokenization moved up a level.** We swapped whitespace/word tokens for **subword tokenizers** (`AutoTokenizer`),
>   and saw why one word can become several tokens.
> - **Full fine-tuning with Hugging Face.** We fine-tuned **DistilBERT** for classification and token-classification
>   using the **`Trainer`** API, mapping each `Trainer` piece back to the manual loop it replaces.
> - **Efficient fine-tuning.** We used **LoRA / PEFT** to train a tiny fraction of the parameters and compared cost vs. quality.
>
> **The through-line:** *from-scratch → `Trainer` → LoRA*, all on **encoder-style** models that **classify or tag**.
>
> **Today** we keep the `Trainer` workflow but move to models that **generate text** - encoder-**decoders** and
> **decoder-only** models - and then focus on *how* that text gets generated (decoding).

## Notebook Map

| Part | Topic | Model | ~min |
|---|---|---|---|
| 0 | Setup & roadmap | - | 5 |
| 1 | Recap: two architectures | tokenizers | 5 |
| **A** | **Fine-tune T5 (encoder-decoder)** on summarization | `t5-small` | 25 |
| ☕ | **Break** | - | 5 |
| **B** | **Fine-tune distilgpt2 (decoder-only)** as a language model | `distilgpt2` | 20 |
| **C** | **Decoding deep-dive** (the main event) | reuse Part B | 25 |
| **D** | Prompting, briefly | `flan-t5-small` | 10 |
| Recap | Wrap-up & take-home exercises | - | 5 |

## How to use this notebook

- **Run it top to bottom.** Later cells reuse models/tokenizers defined earlier - a fresh *Restart & Run All* is the safe way to run it.
- **Use a GPU.** On Colab: *Runtime → Change runtime type → T4 GPU*. Each fine-tune then finishes in ~2-5 minutes. It also runs on CPU, just slowly.
- **First run needs internet** (to pull models, datasets, and metric code); everything is cached afterwards.
- **Call-out boxes:** *Theory Card* (concept), *In practice* (what matters in real work), *Predict first!* (guess before running), **Your turn** (hands-on).
- **`SMOKE_TEST`** (next config cell): set it `True` for a 30-second dry run of the whole notebook on tiny data - handy to check everything is wired before the real run.

## Part 0 - Setup

Install (Colab) and import. The `%%capture` keeps the pip noise out of the way.

In [ ]:
%%capture
# On Colab these are mostly preinstalled; this pins the ones we rely on.
# If a package is missing locally, install it in your environment before running.
!pip install -q -U "transformers>=4.55" "datasets>=3" evaluate rouge_score accelerate ipywidgets

In [ ]:
import random, math, textwrap            # stdlib: RNG seeding, perplexity = exp(loss), pretty-printing
import numpy as np
import torch
import torch.nn.functional as F           # softmax / log-softmax for the from-scratch decoder in Part C

# Hugging Face building blocks, grouped by what they do:
from transformers import (
    AutoTokenizer,                         # text  <-> token ids (auto-picks the right tokenizer per model)
    AutoModelForSeq2SeqLM,                 # encoder-decoder head (T5, BART, Flan-T5)  -> Parts A & D
    AutoModelForCausalLM,                  # decoder-only head (GPT-2, distilgpt2)     -> Parts B & C
    DataCollatorForSeq2Seq,                # dynamic padding + builds decoder labels    (Part A)
    DataCollatorForLanguageModeling,       # copies input_ids -> labels; the model does the shift (Part B)
    Seq2SeqTrainingArguments, Seq2SeqTrainer,   # Trainer variant that can *generate* during eval (Part A)
    TrainingArguments, Trainer,            # the standard training loop wrapper          (Part B)
    set_seed,                              # seeds python/numpy/torch in one call
)
from datasets import load_dataset          # pulls datasets from the Hugging Face Hub
import evaluate                            # metric hub (we use ROUGE for summarization)

# Train on GPU if there is one; fall back to CPU (slower but everything still runs).
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()       # fp16 mixed precision: fast, fine for GPT-2 (Part B)
# T5 is numerically unstable in fp16 (it can emit NaN loss), so Part A prefers bf16 when the GPU
# supports it (Ampere+). On a Colab T4 both are False -> T5 trains in plain fp32, which is safe.
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

def seed_everything(seed=42):
    """Make runs reproducible by seeding every RNG the stack touches.
    We call this again right before sampling demos so their output is repeatable."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)                         # HF's own helper (also seeds its internal RNGs)

seed_everything()
print(f"transformers device: {device} | fp16: {USE_FP16} | bf16: {USE_BF16}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

### One config cell - the dials we turn

Keeping the knobs in one place means every model trains under the same honest, small budget so the
live session stays fast. In a real project these are exactly the values you'd *sweep*.

> **`SMOKE_TEST = True`** shrinks every dataset to a handful of rows and trains for a few steps, so you can
> verify the whole notebook runs end-to-end in well under a minute. Set it back to `False` for the real thing.

In [ ]:
SMOKE_TEST = False   # <- flip to True for a fast dry-run of the entire notebook

# ---- Part A: T5 summarization ----
SUM_TRAIN_SIZE = 1000
SUM_VAL_SIZE   = 200
SUM_EPOCHS     = 1
SUM_MAX_INPUT  = 384    # dialogue tokens
SUM_MAX_TARGET = 64     # summary tokens

# ---- Part B: distilgpt2 language modeling ----
LM_BLOCK_SIZE  = 128
LM_EPOCHS      = 3
LM_MAX_BLOCKS  = None   # None = use the whole tiny corpus

# ---- shared ----
BATCH_SIZE = 8
LR         = 5e-5

if SMOKE_TEST:
    SUM_TRAIN_SIZE, SUM_VAL_SIZE, SUM_EPOCHS = 40, 20, 1
    LM_EPOCHS, BATCH_SIZE, LM_MAX_BLOCKS = 1, 4, 40
    print("SMOKE_TEST on: tiny data, minimal steps.")
print(f"batch={BATCH_SIZE} lr={LR} | summ {SUM_TRAIN_SIZE}/{SUM_VAL_SIZE}x{SUM_EPOCHS}ep | lm {LM_EPOCHS}ep")

## Part 1 - Recap: two families of Transformer

Everything today is one of these two shapes:

| | **Encoder-Decoder** (seq2seq) | **Decoder-only** (causal LM) |
|---|---|---|
| Examples | T5, BART, mT5, Flan-T5 | GPT-2, GPT-4, Llama, Mistral |
| Reads input with | a bidirectional **encoder** | the same left-to-right stack |
| Produces output with | an autoregressive **decoder** that *attends to the encoder* | itself, continuing the prompt |
| Natural fit | transform **X → Y** (translate, summarize, T5's "text-to-text") | *continue* text; open-ended generation |
| Training objective today | teacher-forced seq2seq cross-entropy | next-token cross-entropy |

> **Theory Card - why "text-to-text" (T5).** T5 casts *every* task as "input text → output text". Summarization is
> `"summarize: <doc>" → "<summary>"`; translation is `"translate English to German: ..." → "..."`. Same model, same
> loss, different prefix. We'll use that prefix trick in Part A.

The cleanest way to feel the difference is to watch each model's **tokenizer** turn text into IDs.

In [ ]:
# Run the SAME sentence through BOTH tokenizers and compare. Things to notice:
#   * different vocabularies  -> the sentence splits into different subword pieces
#   * T5 uses a SentencePiece '_' marker for spaces; GPT-2 uses a byte-level 'Ġ' marker
#   * different special tokens -> T5 appends </s>; GPT-2 has no [CLS]/[SEP] at all
demo = "Fine-tuning transformers is surprisingly approachable!"

# from_pretrained downloads (then caches) the tokenizer files that shipped with each model.
t5_tok  = AutoTokenizer.from_pretrained("t5-small")      # encoder-decoder family
gpt_tok = AutoTokenizer.from_pretrained("distilgpt2")    # decoder-only family

for name, tok in [("T5 (encoder-decoder)", t5_tok), ("distilgpt2 (decoder-only)", gpt_tok)]:
    ids = tok(demo)["input_ids"]                 # tokenizer(text) -> dict; "input_ids" are the integer tokens
    pieces = tok.convert_ids_to_tokens(ids)      # map ids back to the human-readable subword strings
    print(f"\n{name}")
    print("  #tokens:", len(ids))                # same sentence, different token counts per tokenizer
    print("  pieces :", pieces)
    print("  eos:", tok.eos_token, "| pad:", tok.pad_token)   # note distilgpt2's pad is None (see below)

**Notice:** distilgpt2 has **no pad token** (`pad = None`). Decoder-only models weren't trained with padding,
so before we batch or generate with them we set `pad_token = eos_token`. We'll do that in Part B. This kind of
tiny detail is exactly what breaks a training script at 2am.

# Part A - Fine-tune T5 (encoder-decoder) on summarization

**Task:** given a short chat dialogue, produce a one-sentence summary. Dataset: **SamSum** (messenger-style
conversations + human summaries) - small and readable, perfect for a live fine-tune.

Our plan is the standard seq2seq recipe:

1. Load the data (a subset, to stay fast).
2. Turn `"summarize: <dialogue>" → "<summary>"` into token ids (`text_target=` for the labels).
3. Hand it to `Seq2SeqTrainer` with a `DataCollatorForSeq2Seq`.
4. Score with **ROUGE** and eyeball a few generations.

> **Theory → Code**
>
> | Concept | Object |
> |---|---|
> | task prefix | `"summarize: "` prepended to inputs |
> | encoder input | `input_ids` (the dialogue) |
> | decoder target | `labels` (the summary), built with `text_target=` |
> | teacher forcing | handled inside the model + collator |
> | metric | `evaluate.load("rouge")` |

In [ ]:
# Load SamSum from a *parquet* mirror. (The original script-based `samsum` loader no longer
# works: datasets>=4 dropped remote dataset scripts, which is why we point at this Hub copy.)
raw_sum = load_dataset("knkarthick/samsum")               # DatasetDict with train/validation/test splits
raw_sum = raw_sum.filter(lambda ex: ex["dialogue"] and ex["summary"])   # guard against empty/None rows

# Keep only a subset so a live fine-tune finishes in minutes. .shuffle then .select(range(N))
# gives a random-but-reproducible slice (seed fixes which rows we get).
train_sum = raw_sum["train"].shuffle(seed=42).select(range(SUM_TRAIN_SIZE))
val_sum   = raw_sum["validation"].shuffle(seed=42).select(range(SUM_VAL_SIZE))
print(train_sum)                                          # shows features (dialogue, summary, id) and row count

# Peek at one example so we know exactly what X (dialogue) and Y (summary) look like.
ex = train_sum[0]
print("\nDIALOGUE:\n", textwrap.fill(ex["dialogue"], 90))
print("\nSUMMARY:\n", ex["summary"])

In [ ]:
model_name_t5 = "t5-small"                                       # ~60M params: small enough for a live demo
t5_tokenizer = AutoTokenizer.from_pretrained(model_name_t5)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(model_name_t5)   # loads pretrained encoder + decoder weights

PREFIX = "summarize: "   # T5 was multitask-trained with task tags; this prefix tells it which job to do

def preprocess_sum(batch):
    """Turn a batch of {dialogue, summary} into model-ready token ids.
    Encoder side -> input_ids (the prefixed dialogue); decoder side -> labels (the summary)."""
    inputs = [PREFIX + d for d in batch["dialogue"]]             # prepend the task tag to every dialogue
    # Tokenize the ENCODER inputs. truncation caps very long dialogues at SUM_MAX_INPUT tokens.
    model_inputs = t5_tokenizer(inputs, max_length=SUM_MAX_INPUT, truncation=True)
    # Tokenize the TARGETS. `text_target=` tells the tokenizer to use its target-side settings,
    # which for T5 is the same vocab but is the idiomatic, future-proof way to encode labels.
    labels = t5_tokenizer(text_target=batch["summary"], max_length=SUM_MAX_TARGET, truncation=True)
    model_inputs["labels"] = labels["input_ids"]                 # the Trainer computes loss against these
    return model_inputs

# .map(batched=True) runs preprocess_sum over chunks of rows; remove_columns drops the raw text
# so the dataset that reaches the model contains only tensors (input_ids, attention_mask, labels).
tok_train_sum = train_sum.map(preprocess_sum, batched=True, remove_columns=train_sum.column_names)
tok_val_sum   = val_sum.map(preprocess_sum,   batched=True, remove_columns=val_sum.column_names)
print("one tokenized example keys:", list(tok_train_sum[0].keys()))
print("input_ids len:", len(tok_train_sum[0]["input_ids"]), "| labels len:", len(tok_train_sum[0]["labels"]))

**The collator does the padding *per batch*** (dynamic padding), and crucially pads the `labels` with `-100`
so those positions are ignored by the loss - exactly the ignore-index trick from last session, now for free.

In [ ]:
# The collator pads each batch to its own longest row (dynamic padding = less wasted compute) and,
# crucially, pads the LABELS with -100 so those positions are ignored by the cross-entropy loss.
data_collator_sum = DataCollatorForSeq2Seq(t5_tokenizer, model=t5_model)

rouge = evaluate.load("rouge")   # ROUGE = n-gram overlap between generated and reference summaries

def compute_rouge(eval_pred):
    """Called by the Trainer after each eval. Receives generated token ids + gold labels,
    decodes both back to text, and returns ROUGE scores (scaled to 0-100 for readability)."""
    preds, labels = eval_pred
    if isinstance(preds, tuple):          # some model configs return (logits, ...) -> keep the token ids
        preds = preds[0]
    # Replace the -100 padding with a real pad id BEFORE decoding, otherwise batch_decode chokes on -100.
    preds  = np.where(preds  != -100, preds,  t5_tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, t5_tokenizer.pad_token_id)
    decoded_preds  = t5_tokenizer.batch_decode(preds,  skip_special_tokens=True)   # ids -> text
    decoded_labels = t5_tokenizer.batch_decode(labels, skip_special_tokens=True)
    # use_stemmer=True so "runs"/"running" count as a match; rouge.compute returns fractions in [0,1].
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}

### Baseline first: what does *un*-fine-tuned t5-small say?

Always look before you train. t5-small has seen the word "summarize", but it has never seen SamSum chats.

In [ ]:
def summarize_t5(dialogue, model, max_length=SUM_MAX_TARGET, num_beams=4):
    """Generate a summary for one dialogue. num_beams=4 -> beam search, a good default for
    constrained tasks like summarization (we dig into decoding choices in Part C)."""
    model.eval()                                          # eval mode: disables dropout, etc.
    inputs = t5_tokenizer(PREFIX + dialogue, return_tensors="pt",   # "pt" -> PyTorch tensors
                          truncation=True, max_length=SUM_MAX_INPUT).to(model.device)
    with torch.no_grad():                                 # no gradients needed at inference -> less memory
        # max_length caps the *generated* summary length. (We pass max_length rather than
        # max_new_tokens so we don't set both at once, which triggers a noisy HF warning.)
        out = model.generate(**inputs, max_length=max_length, num_beams=num_beams)
    return t5_tokenizer.decode(out[0], skip_special_tokens=True)    # ids -> text, drop </s>/<pad>

sample = val_sum[0]                                        # a held-out example the model won't train on
t5_model.to(device)
print("DIALOGUE:\n", textwrap.fill(sample["dialogue"], 90))
print("\nGOLD    :", sample["summary"])                   # the human reference summary
print("BASELINE:", summarize_t5(sample["dialogue"], t5_model))   # what t5-small says BEFORE fine-tuning

In [ ]:
training_args_sum = Seq2SeqTrainingArguments(
    output_dir="t5-samsum",
    num_train_epochs=SUM_EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    predict_with_generate=True,          # eval by *generating*, so ROUGE is meaningful
    generation_max_length=SUM_MAX_TARGET,
    fp16=False,                          # NOT fp16: T5 goes NaN in fp16 (a classic footgun)
    bf16=USE_BF16,                       # bf16 is safe for T5 when available; else fp32
    eval_strategy="epoch",
    logging_steps=25,
    save_strategy="no",
    report_to="none",
)

trainer_sum = Seq2SeqTrainer(
    model=t5_model,
    args=training_args_sum,
    train_dataset=tok_train_sum,
    eval_dataset=tok_val_sum,
    data_collator=data_collator_sum,
    compute_metrics=compute_rouge,
)
trainer_sum.train()

In [ ]:
print("Final validation ROUGE:")
metrics = trainer_sum.evaluate()
for k, v in metrics.items():
    if "rouge" in k:
        print(f"  {k}: {v}")

### After fine-tuning - same dialogue, better summary?

In [ ]:
print("DIALOGUE:\n", textwrap.fill(sample["dialogue"], 90))
print("\nGOLD      :", sample["summary"])
print("FINE-TUNED:", summarize_t5(sample["dialogue"], t5_model))

# a couple more from the validation set
for i in [1, 2, 3]:
    s = val_sum[i]
    print("\n---")
    print("GOLD      :", s["summary"])
    print("FINE-TUNED:", summarize_t5(s["dialogue"], t5_model))

> **Your turn (Part A).**
> 1. Bump `SUM_MAX_TARGET` to 96 or `SUM_EPOCHS` to 2 in the config cell and re-run this part. Does ROUGE move? Do summaries get longer?
> 2. Feed `summarize_t5` a dialogue you write yourself.
> 3. *Discuss:* ROUGE rewards n-gram overlap with the gold summary. Name one good summary it would unfairly punish.

---
## ☕ Break (5 min)

Stand up, stretch. When we come back: the **other** kind of Transformer.

*While you wait - a question to chew on:* T5 needed a `labels` column (the target summary).
The next model, distilgpt2, will train on **plain text with no labels at all**. Where do its training targets come from?

# Part B - Fine-tune distilgpt2 (decoder-only) as a language model

No input/output pairs here - just **text**. A causal LM learns to predict the **next token** given all previous
ones, so the training signal at position *t* is simply the token at position *t+1*. `DataCollatorForLanguageModeling`
just copies `input_ids` into `labels`; the **model** then shifts internally (it scores its prediction after token
*t* against token *t+1*), so we don't build the shift by hand.

We'll fine-tune on **tiny-shakespeare** so the effect is obvious: a general-purpose distilgpt2 will start
writing in mock-Elizabethan English. Then in Part C this exact model becomes our playground for decoding.

> **That break question, answered:** the targets *are the text itself, shifted by one*. This is why LMs are called
> *self-supervised* - the data labels itself.

In [ ]:
# tiny-shakespeare from a parquet mirror (again: script-based loaders are gone in datasets>=4).
# It's just raw text under a single 'Text' column - no labels, because a language model makes its own.
raw_lm = load_dataset("Trelis/tiny-shakespeare")
print(raw_lm)                                    # note: only a 'Text' feature, no target column
print("\nsample:\n", raw_lm["train"][0]["Text"][:300])

In [ ]:
model_name_gpt = "distilgpt2"                              # a small 6-layer GPT-2 (~82M params)
gpt2_tokenizer = AutoTokenizer.from_pretrained(model_name_gpt)
# GPT-2 was trained without padding, so its tokenizer has pad_token=None. Batching and generation
# both need *some* pad id, so we reuse the end-of-sequence token as the pad token (a standard fix).
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = AutoModelForCausalLM.from_pretrained(model_name_gpt).to(device)

def tokenize_lm(batch):
    """Just convert text -> input_ids. No max_length/truncation here: we keep every token
    and re-chunk into fixed windows in the next step."""
    return gpt2_tokenizer(batch["Text"])

# Map over the corpus and drop the raw 'Text' column, leaving token ids we can concatenate.
tok_lm = raw_lm.map(tokenize_lm, batched=True, remove_columns=raw_lm["train"].column_names)

**Chunking into blocks.** The corpus is a few long strings. We concatenate all tokens and slice them into
fixed `LM_BLOCK_SIZE` windows - the standard causal-LM preprocessing. Each block is one training example.

In [ ]:
def group_texts(batch):
    """Standard causal-LM packing: glue all token ids into one long stream, then slice it into
    equal-length blocks. Each block becomes one training example the model reads left-to-right."""
    concatenated = sum(batch["input_ids"], [])            # flatten list-of-lists -> one long token list
    total_len = (len(concatenated) // LM_BLOCK_SIZE) * LM_BLOCK_SIZE   # drop the ragged remainder
    # cut [0:128], [128:256], ... so every block is exactly LM_BLOCK_SIZE tokens
    input_ids = [concatenated[i:i + LM_BLOCK_SIZE] for i in range(0, total_len, LM_BLOCK_SIZE)]
    # attention_mask is all 1s: every position is a real token (no padding inside a packed block)
    return {"input_ids": input_ids, "attention_mask": [[1] * LM_BLOCK_SIZE for _ in input_ids]}

lm_dataset = tok_lm.map(group_texts, batched=True)
if LM_MAX_BLOCKS:                                    # keep the dry-run fast
    n = min(LM_MAX_BLOCKS, len(lm_dataset["train"]))
    lm_dataset["train"] = lm_dataset["train"].select(range(n))
print("train blocks:", len(lm_dataset["train"]), "| each of length", LM_BLOCK_SIZE)

# mlm=False => causal LM. The collator sets labels = input_ids (a copy, with pad -> -100); the MODEL
# shifts internally so the prediction after token t is scored against token t+1. (mlm=True would instead
# mask random tokens for BERT-style training.)
data_collator_lm = DataCollatorForLanguageModeling(gpt2_tokenizer, mlm=False)

### Before fine-tuning: distilgpt2's default voice

In [ ]:
def gpt2_generate(prompt, model, max_new_tokens=40, **gen_kwargs):
    """Continue `prompt` with the model. **gen_kwargs is where the Part-C decoding controls go
    (do_sample, temperature, top_k, top_p, num_beams, ...); passing none = greedy defaults."""
    model.eval()
    ids = gpt2_tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens,
                             pad_token_id=gpt2_tokenizer.eos_token_id,   # silences a pad-token warning
                             **gen_kwargs)
    return gpt2_tokenizer.decode(out[0], skip_special_tokens=True)

PROMPT = "To be, or not to be,"                                 # reused throughout Parts B and C
# do_sample=False -> greedy decoding, so this baseline output is deterministic.
print("BEFORE fine-tuning:\n", gpt2_generate(PROMPT, gpt2_model, do_sample=False))

In [ ]:
training_args_lm = TrainingArguments(
    output_dir="distilgpt2-shakespeare",
    num_train_epochs=LM_EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    fp16=USE_FP16,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
)

trainer_lm = Trainer(
    model=gpt2_model,
    args=training_args_lm,
    train_dataset=lm_dataset["train"],
    data_collator=data_collator_lm,
)
trainer_lm.train()

### After fine-tuning: same prompt, new accent

We report **perplexity** = `exp(loss)`: roughly "how many tokens the model is choosing between on average".
Lower is a more confident language model.

In [ ]:
# grab the last recorded training loss (logging steps use "loss"; the final summary uses "train_loss")
losses = [h.get("loss", h.get("train_loss")) for h in trainer_lm.state.log_history]
final_loss = [l for l in losses if l is not None][-1]
print(f"final train loss: {final_loss:.3f}  ->  perplexity ~ {math.exp(final_loss):.1f}\n")

print("AFTER fine-tuning (greedy):\n", gpt2_generate(PROMPT, gpt2_model, do_sample=False))

It won't be Shakespeare - it's a tiny model on a tiny corpus for a couple of minutes - but the **register**
should have shifted noticeably toward the training text. That shift is the whole point: fine-tuning moved the
distribution. **Now let's control how we sample from that distribution.**

# Part C - Decoding: turning probabilities into text  ⭐

A language model does **not** output text. At each step it outputs a vector of **logits** - one score per
vocabulary token - which we turn into a probability distribution over "what comes next". **Decoding** is the
algorithm that repeatedly picks a token from that distribution and feeds it back in.

Same model, same weights - but the decoding choice changes everything from *"safe and repetitive"* to *"creative and unhinged"*.

We'll do this in two passes:

- **C1 - By hand.** Implement greedy, temperature, top-k, and top-p from raw logits, so there is no magic left.
- **C2 - With `generate`.** The production API, one cell per strategy, side by side - including **beam search**.

## C1 - Decoding from scratch

The engine is one function: given the model's next-token logits, return **one** chosen token id. Everything
else (greedy vs sampling vs top-k vs top-p) is just *how we massage those logits before picking*.

> **Theory Card - the four knobs**
> - **Greedy:** take the argmax. Deterministic. Safe, often dull, prone to loops.
> - **Temperature `T`:** divide logits by `T` before softmax. `T<1` sharpens (bolder argmax), `T>1` flattens (more random). `T→0` = greedy.
> - **Top-k:** keep only the `k` highest-prob tokens, renormalize, sample. Cuts the long tail.
> - **Top-p (nucleus):** keep the smallest set of tokens whose probabilities *sum to `p`*, renormalize, sample. The set size adapts to how peaked the distribution is.

In [ ]:
def pick_next_token(logits, do_sample=False, temperature=1.0, top_k=None, top_p=None):
    """logits: 1D tensor [vocab]. Returns a single token id (int)."""
    logits = logits.clone().float()

    if not do_sample:                      # ---- greedy ----
        return int(torch.argmax(logits))

    logits = logits / max(temperature, 1e-6)   # ---- temperature ----

    if top_k is not None:                  # ---- top-k: keep k largest, mask the rest ----
        kth = torch.topk(logits, top_k).values[-1]
        logits[logits < kth] = float("-inf")

    if top_p is not None:                  # ---- top-p / nucleus ----
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        # mask everything AFTER we first cross p (shift keeps at least one token)
        remove = cum_probs > top_p
        remove[..., 1:] = remove[..., :-1].clone()
        remove[..., 0] = False
        logits[sorted_idx[remove]] = float("-inf")

    probs = F.softmax(logits, dim=-1)      # ---- sample from what survives ----
    return int(torch.multinomial(probs, num_samples=1))


@torch.no_grad()
def generate_by_hand(prompt, max_new_tokens=40, **kw):
    gpt2_model.eval()
    ids = gpt2_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(max_new_tokens):
        logits = gpt2_model(ids).logits[0, -1, :]      # next-token logits for the last position
        nxt = pick_next_token(logits, **kw)
        ids = torch.cat([ids, torch.tensor([[nxt]], device=device)], dim=1)
        if nxt == gpt2_tokenizer.eos_token_id:
            break
    return gpt2_tokenizer.decode(ids[0], skip_special_tokens=True)

### Watch one step: what the knobs do to the *distribution*

Before generating whole passages, let's inspect the top candidates for the **next single token** after our
prompt, and how temperature reshapes them.

In [ ]:
@torch.no_grad()
def show_next_token_dist(prompt, temperature=1.0, top=10):
    """Print the model's top-`top` candidate next tokens and their probabilities as a bar chart.
    Changing `temperature` reshapes the SAME logits: <1 sharpens toward the top token, >1 flattens."""
    ids = gpt2_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    logits = gpt2_model(ids).logits[0, -1, :] / temperature   # [0, -1, :] = logits for the *next* token
    probs = F.softmax(logits, dim=-1)                         # logits -> probability distribution
    vals, idx = torch.topk(probs, top)                        # the `top` most likely tokens
    print(f"prompt={prompt!r}  temperature={temperature}")
    for p, i in zip(vals.tolist(), idx.tolist()):
        bar = "█" * int(p * 50)                               # crude text bar, length ∝ probability
        print(f"  {gpt2_tokenizer.decode([i])!r:<14} {p:6.3f} {bar}")

show_next_token_dist(PROMPT, temperature=1.0)
print()
show_next_token_dist(PROMPT, temperature=0.5)   # sharper
print()
show_next_token_dist(PROMPT, temperature=1.5)   # flatter

> **Predict first!** Before running the next cell: which setting will be the most *repetitive*, and which the most *random*?

In [ ]:
seed_everything(0)
print("GREEDY (argmax):")
print(" ", generate_by_hand(PROMPT, do_sample=False), "\n")

print("TEMPERATURE = 0.7 sampling:")
print(" ", generate_by_hand(PROMPT, do_sample=True, temperature=0.7), "\n")

print("TEMPERATURE = 1.5 sampling (wild):")
print(" ", generate_by_hand(PROMPT, do_sample=True, temperature=1.5), "\n")

print("TOP-K = 20:")
print(" ", generate_by_hand(PROMPT, do_sample=True, top_k=20), "\n")

print("TOP-P = 0.9 (nucleus):")
print(" ", generate_by_hand(PROMPT, do_sample=True, top_p=0.9))

## C2 - The same strategies via `model.generate`

In practice you never hand-roll the loop - `generate` is faster, batched, and supports beam search. But now you
know *exactly* what each argument does. Here is the full menu on one prompt.

> **In practice.** `generate`'s defaults changed across versions and are model-dependent. Be **explicit**:
> pass `do_sample`, `temperature`, `top_k`, `top_p`, `num_beams` yourself rather than trusting defaults.

In [ ]:
def show(title, text):
    print(f"### {title}\n" + textwrap.fill(text, 90) + "\n")

seed_everything(0)
show("Greedy",
     gpt2_generate(PROMPT, gpt2_model, do_sample=False))

show("Beam search (num_beams=5, no_repeat_ngram_size=2)",
     gpt2_generate(PROMPT, gpt2_model, num_beams=5, no_repeat_ngram_size=2,
                   early_stopping=True))

show("Pure sampling (temperature=1.0)",
     gpt2_generate(PROMPT, gpt2_model, do_sample=True, temperature=1.0, top_k=0))

show("Top-k = 50",
     gpt2_generate(PROMPT, gpt2_model, do_sample=True, top_k=50))

show("Top-p = 0.92 (nucleus)",
     gpt2_generate(PROMPT, gpt2_model, do_sample=True, top_p=0.92, top_k=0))

show("Combined: temp=0.8, top_k=50, top_p=0.95",
     gpt2_generate(PROMPT, gpt2_model, do_sample=True, temperature=0.8, top_k=50, top_p=0.95))

### Beam search's failure mode: repetition

Beam search maximizes sequence probability, which for open-ended generation often means **looping** ("I don't
know. I don't know. I don't know."). `no_repeat_ngram_size` is a band-aid. This is *why* open-ended chat/story
models sample (top-p) instead of using beams - beams are better for *constrained* tasks like translation and
summarization (that's why Part A used `num_beams=4`).

In [ ]:
show("Beam WITHOUT no_repeat_ngram (watch it loop)",
     gpt2_generate(PROMPT, gpt2_model, num_beams=5, max_new_tokens=50))

### Your turn (Part C) - interactive knobs

Run the slider cell (needs `ipywidgets`; on Colab it just works). Drag temperature / top-k / top-p and watch the
text change live. If widgets aren't available, use the plain loop cell below it.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def _interactive(prompt="To be, or not to be,", temperature=0.8, top_k=50, top_p=0.95, seed=0):
        seed_everything(seed)
        txt = gpt2_generate(prompt, gpt2_model, do_sample=True,
                            temperature=temperature, top_k=top_k, top_p=top_p, max_new_tokens=50)
        print(textwrap.fill(txt, 90))

    widgets.interact(
        _interactive,
        prompt=widgets.Text(value="To be, or not to be,"),
        temperature=widgets.FloatSlider(min=0.1, max=1.8, step=0.1, value=0.8),
        top_k=widgets.IntSlider(min=0, max=100, step=5, value=50),
        top_p=widgets.FloatSlider(min=0.1, max=1.0, step=0.05, value=0.95),
        seed=widgets.IntSlider(min=0, max=20, step=1, value=0),
    )
except Exception as e:
    print("ipywidgets unavailable ->", e)

In [ ]:
# Plain fallback: sweep temperature by hand.
seed_everything(0)
for T in [0.3, 0.7, 1.0, 1.4]:
    print(f"--- temperature={T} ---")
    print(textwrap.fill(gpt2_generate(PROMPT, gpt2_model, do_sample=True, temperature=T,
                                      top_k=0, max_new_tokens=40), 90), "\n")

> **Discuss.**
> 1. You need reproducible outputs for a demo. Which strategy, and what else must you fix? *(hint: the seed.)*
> 2. Top-k=1 is equivalent to which strategy? What about temperature → 0?
> 3. Why does top-p adapt better than top-k when the model is *very sure* about the next token?

# Part D - Prompting, briefly

Fine-tuning changes the **weights**. Prompting changes only the **input** - no training. It works best on models
that were **instruction-tuned**. We'll use **`flan-t5-small`**: the same T5 architecture from Part A, but further
trained to follow instructions. (Base `distilgpt2` is *not* instruction-tuned, which is why plain prompts confuse it.)

In [ ]:
# Flan-T5 = T5 further trained on a huge mix of *instructions*. Same seq2seq architecture as Part A,
# but it has learned to treat the input as a command to follow rather than text to continue.
flan_name = "google/flan-t5-small"
flan_tok = AutoTokenizer.from_pretrained(flan_name)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(flan_name).to(device)

def ask_flan(prompt, max_new_tokens=50, **kw):
    """Send a raw instruction/prompt straight to Flan-T5 and return its answer. No fine-tuning here -
    the model is frozen; we only change the *text* we feed it."""
    ids = flan_tok(prompt, return_tensors="pt", truncation=True).to(device)
    out = flan_model.generate(**ids, max_new_tokens=max_new_tokens, **kw)   # greedy by default
    return flan_tok.decode(out[0], skip_special_tokens=True)

### Zero-shot: just ask

No examples, no training - we phrase the task as an instruction and read the answer. Watch the **last** prompt:
instead of asking the model to *recall* a fact (a small model will happily get that wrong), we put the fact in
the prompt and ask it to *use* the context. Grounding answers in provided text is exactly the idea behind
**retrieval-augmented generation (RAG)**.

In [ ]:
for p in [
    "Translate to German: I love natural language processing.",
    "What is the sentiment of this review? 'The plot dragged and I fell asleep.'",
    "Summarize: A group of friends plan a surprise party, argue about the cake, and finally agree on chocolate.",
    # Context-grounded QA: we hand the model the facts, then ask. Small models are unreliable at
    # closed-book recall (try 'What is the capital of France?' -> it may say 'london'!), but they
    # answer well when the answer is *in the prompt*. This is the seed of retrieval-augmented generation.
    "Answer the question using the context. Context: Marie Curie was born in Warsaw, Poland, in 1867. "
    "Question: Where was Marie Curie born?",
]:
    print("PROMPT :", p)
    print("FLAN   :", ask_flan(p), "\n")

### Few-shot: teach the format with examples

For a small topic-classification task we show 2 labelled examples, then ask for a third. The model
generalizes the *pattern and the output format* from the prompt alone - no weight update.

In [ ]:
few_shot = textwrap.dedent("""\
    Classify the topic as SPORT, TECH, or FOOD.
    Text: The new GPU doubles training speed. -> TECH
    Text: He scored a hat-trick in the final. -> SPORT
    Text: This ramen broth simmers for 12 hours. ->""")
print(few_shot)
print("\nFLAN ->", ask_flan(few_shot, max_new_tokens=5))

### Prompt phrasing matters

Same task, two phrasings. Small instruction-tuned models are **sensitive** to wording - a real source of
frustration and a real lever.

In [ ]:
q = "The movie was a complete waste of two hours."
for phrasing in [
    f"Is this review positive or negative? '{q}'",
    f"Review: '{q}'. Sentiment (positive/negative):",
]:
    print("PROMPT :", phrasing)
    print("FLAN   :", ask_flan(phrasing, max_new_tokens=5), "\n")

> **Your turn (Part D).**
> 1. Add a **third** example to the few-shot prompt and a harder test text.
> 2. Run the same zero-shot prompts through the *base* distilgpt2 (`gpt2_generate`). Why is it so much worse at *following* the instruction?
> 3. When would you fine-tune (Parts A/B) instead of prompt? Jot two deciding factors.

# Recap & take-home

**What we did**

| | Model | Job | Objective | How we decoded |
|---|---|---|---|---|
| Part A | t5-small | summarization | seq2seq cross-entropy | beam search |
| Part B | distilgpt2 | language modeling | next-token cross-entropy | (set up for Part C) |
| Part C | (Part B model) | generation | - | greedy / beam / temp / top-k / top-p |
| Part D | flan-t5-small | anything, via text | none (frozen) | prompting |

**The three big ideas**

1. **Architecture follows the task.** Transform X→Y ⇒ encoder-decoder; continue/generate ⇒ decoder-only.
2. **Fine-tuning is a recipe, not magic:** tokenize → collate (with the right label handling) → `Trainer` → a metric you trust.
3. **Decoding is a dial you own.** The *same weights* give safe-and-boring or creative-and-risky depending on greedy vs sampling and the temperature / top-k / top-p you choose. Beams for constrained tasks; nucleus for open-ended ones.

**Take-home exercises**

1. Swap Part A to **translation**: prefix `"translate English to German: "` and the `opus_books` (en-de) dataset. What changes, what doesn't?
2. Fine-tune Part B on a corpus *you* like (song lyrics, code, your notes) and find the temperature that reads best.
3. Implement **repetition_penalty** inside `pick_next_token` and compare it to `no_repeat_ngram_size`.
4. Measure it: compute ROUGE of `flan-t5-small` *zero-shot* on SamSum vs your fine-tuned t5-small from Part A. Is fine-tuning worth it here?

*Great work - see you next session.*